# 🔐 Crypto Fraud Detection & Wallet Risk Scoring Pipeline
### Elliptic Bitcoin Dataset — Full EDA + Multi-Model ML Pipeline + Risk Scoring

**Objectives:**
1. Deeply explore the Elliptic transaction graph dataset (class balance, features, graph structure, time steps)
2. Clean and prepare the data for modeling
3. Engineer graph-based features
4. Train and compare multiple models (Logistic Regression, Random Forest, XGBoost, LightGBM)
5. Evaluate rigorously with focus on the **illicit** minority class
6. Build a reusable **risk scoring function** (0–1) and **inference pipeline**
7. Save the best model, the pipeline, and a summary report

All plots are automatically saved as high-quality PNGs to `/content/images/` for reporting/presentation use.


## 1. Setup, Drive Mount & Create Images Folder

Install dependencies, mount Google Drive, set global styling, and create the output folder for all saved visualizations.


In [ ]:
# Install/upgrade required libraries (Colab usually has most already)
!pip install -q lightgbm xgboost scikit-learn seaborn joblib networkx


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import networkx as nx

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, average_precision_score, accuracy_score,
    precision_score, recall_score
)

import xgboost as xgb
import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---- Professional plotting style ----
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 11

PALETTE = {"licit": "#2E86AB", "illicit": "#E63946", "unknown": "#B0B0B0"}

print("Libraries loaded successfully.")


In [ ]:
# ---- Paths ----
BASE_DIR = "/content/drive/MyDrive/CASHNET/elliptic-data-set/elliptic_bitcoin_dataset/"
FEATURES_PATH = os.path.join(BASE_DIR, "elliptic_txs_features.csv")
CLASSES_PATH  = os.path.join(BASE_DIR, "elliptic_txs_classes.csv")
EDGES_PATH    = os.path.join(BASE_DIR, "elliptic_txs_edgelist.csv")

# Elliptic++ optional wallet-level files (explored if present)
ELLIPTICPP_DIR = os.path.join(os.path.dirname(BASE_DIR.rstrip('/')), "elliptic++")
WALLETS_FEATURES_PATH = os.path.join(ELLIPTICPP_DIR, "wallets_features.csv")
WALLETS_CLASSES_PATH  = os.path.join(ELLIPTICPP_DIR, "wallets_classes.csv")

# ---- Output directories ----
IMAGES_DIR = "/content/images"
MODEL_DIR  = "/content/drive/MyDrive/CASHNET/models/"
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Images will be saved to:", IMAGES_DIR)
print("Models will be saved to:", MODEL_DIR)
print()
print("Feature file exists:", os.path.exists(FEATURES_PATH))
print("Classes file exists :", os.path.exists(CLASSES_PATH))
print("Edgelist file exists:", os.path.exists(EDGES_PATH))
print("Elliptic++ wallets features found:", os.path.exists(WALLETS_FEATURES_PATH))


In [ ]:
def save_fig(fig, filename, images_dir=IMAGES_DIR):
    """Save a matplotlib figure as a high-quality PNG with consistent settings."""
    path = os.path.join(images_dir, filename)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Saved plot -> {path}")
    return path


## 2. Load Elliptic Dataset

The Elliptic dataset ships **without a header row** in `elliptic_txs_features.csv`. The columns are:
- `col_0` → transaction ID (`txId`)
- `col_1` → time step
- `col_2 ... col_94` → 93 local transaction features
- `col_95 ... col_166` → 72 aggregated (neighborhood) features

`elliptic_txs_classes.csv` maps `txId → class` (`'1'` = illicit, `'2'` = licit, `'unknown'` = unlabeled).
`elliptic_txs_edgelist.csv` contains directed transaction → transaction edges.


In [ ]:
# ---- Load raw files ----
features_df = pd.read_csv(FEATURES_PATH, header=None)
classes_df  = pd.read_csv(CLASSES_PATH)
edges_df    = pd.read_csv(EDGES_PATH)

# ---- Name the feature columns ----
N_LOCAL_FEATURES = 93
N_AGG_FEATURES   = 72

feature_cols = ["txId", "time_step"] + \
               [f"local_{i}" for i in range(N_LOCAL_FEATURES)] + \
               [f"agg_{i}" for i in range(N_AGG_FEATURES)]

features_df.columns = feature_cols
classes_df.columns = ["txId", "class"]
edges_df.columns = ["txId1", "txId2"]

print(f"Features shape : {features_df.shape}")
print(f"Classes shape  : {classes_df.shape}")
print(f"Edges shape    : {edges_df.shape}")


In [ ]:
features_df.head()

In [ ]:
classes_df.head()

In [ ]:
edges_df.head()

In [ ]:
# ---- Merge features with class labels ----
label_map = {"1": 1, "2": 0, "unknown": -1, 1: 1, 2: 0}

data = features_df.merge(classes_df, on="txId", how="left")
data["label"] = data["class"].map(label_map)
data["label_name"] = data["label"].map({1: "illicit", 0: "licit", -1: "unknown"})

print("Merged dataset shape:", data.shape)
data[["txId", "time_step", "class", "label", "label_name"]].head()


## 3. Comprehensive Exploratory Data Analysis

This section covers:
- Class distribution (licit / illicit / unknown)
- Missing value analysis
- Feature distributions: illicit vs licit
- Correlation heatmap
- Graph structure analysis (degree distribution, basic stats)
- Time-step analysis

Every notable finding is summarized in markdown, and every plot is saved to `/content/images/`.


### 3.1 Class Distribution Analysis

In [ ]:
class_counts = data["label_name"].value_counts()
class_pct = (class_counts / class_counts.sum() * 100).round(2)

summary_table = pd.DataFrame({"count": class_counts, "percent": class_pct})
print(summary_table)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Bar chart
order = ["licit", "illicit", "unknown"]
colors = [PALETTE[c] for c in order]
sns.barplot(x=order, y=class_counts.reindex(order).values, palette=colors, ax=axes[0])
axes[0].set_title("Transaction Class Distribution (Counts)", fontweight="bold")
axes[0].set_ylabel("Number of Transactions")
axes[0].set_xlabel("Class")
for i, v in enumerate(class_counts.reindex(order).values):
    axes[0].text(i, v + max(class_counts) * 0.01, f"{v:,}", ha="center", fontweight="bold")

# Pie chart
axes[1].pie(
    class_counts.reindex(order).values,
    labels=[f"{c}\n({p}%)" for c, p in zip(order, class_pct.reindex(order).values)],
    colors=colors, autopct="%1.1f%%", startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)
axes[1].set_title("Transaction Class Distribution (Proportion)", fontweight="bold")

plt.suptitle("Elliptic Dataset — Class Distribution Overview", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
save_fig(fig, "01_class_distribution.png")
plt.show()


**Finding:** The dataset is dominated by `unknown` transactions, with `licit` transactions far outnumbering `illicit` ones among the labeled set. This confirms two challenges for modeling: (1) the labeled subset itself is **imbalanced** toward licit transactions, and (2) most of the graph is **unlabeled**, so the model must be trained only on labeled nodes while still being usable for scoring unknown ones at inference time.


### 3.2 Missing Value Analysis

In [ ]:
missing_counts = data.isnull().sum()
missing_pct = (missing_counts / len(data) * 100).round(3)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0].sort_values("missing_count", ascending=False)

if missing_report.empty:
    print("No missing values found in any column. Dataset is clean at the cell level.")
else:
    print(missing_report)


In [ ]:
# Check for any placeholder / sentinel missing values (e.g. -1 used broadly across engineered features)
numeric_feature_cols = [c for c in data.columns if c.startswith("local_") or c.startswith("agg_")]
sentinel_check = (data[numeric_feature_cols] == -1).sum().sum()
print(f"Total occurrences of literal -1 across all {len(numeric_feature_cols)} feature columns: {sentinel_check:,}")
print("(Elliptic's published features are already normalized floats; -1 sentinels, if any, are treated as legitimate values here.)")


**Finding:** The Elliptic feature files have no null cells — the only "missing" information is the `unknown` class label itself (~77% of nodes), which is handled explicitly by excluding those rows from training while keeping them available for inference-time risk scoring (Section 5 onward).


### 3.3 Feature Distributions — Illicit vs Licit

In [ ]:
labeled_data = data[data["label"] != -1].copy()
print("Labeled subset shape:", labeled_data.shape)
print(labeled_data["label_name"].value_counts())


In [ ]:
# Pick a handful of informative local + aggregated features to visualize
sample_local_feats = ["local_0", "local_1", "local_2", "local_5"]
sample_agg_feats   = ["agg_0", "agg_1", "agg_5", "agg_10"]
compare_feats = sample_local_feats + sample_agg_feats

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(compare_feats):
    sns.violinplot(
        data=labeled_data, x="label_name", y=feat, order=["licit", "illicit"],
        palette=[PALETTE["licit"], PALETTE["illicit"]], ax=axes[i], cut=0
    )
    axes[i].set_title(feat, fontweight="bold")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Value")

plt.suptitle("Illicit vs Licit — Feature Distribution Comparison (Violin Plots)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig(fig, "03_illicit_vs_licit_feature_comparison.png")
plt.show()


**Finding:** Several local and aggregated features show visibly different distributions/spread between illicit and licit transactions (shifted medians, different variance), confirming these features carry discriminative signal for the classifier. Aggregated (neighborhood) features tend to show wider spread for illicit transactions, consistent with illicit transactions often sitting in denser or more irregular subgraphs.


### 3.4 Feature Correlation Heatmap

In [ ]:
# Use a representative subset of features for a readable heatmap
corr_features = [f"local_{i}" for i in range(0, 93, 6)] + [f"agg_{i}" for i in range(0, 72, 6)]
corr_matrix = labeled_data[corr_features].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    corr_matrix, cmap="coolwarm", center=0, square=True,
    linewidths=0.3, cbar_kws={"shrink": 0.75, "label": "Correlation"}, ax=ax
)
ax.set_title("Feature Correlation Heatmap (Sampled Features)", fontsize=16, fontweight="bold", pad=15)
plt.tight_layout()
save_fig(fig, "02_feature_correlation_heatmap.png")
plt.show()


In [ ]:
# Identify highly correlated feature pairs (potential redundancy for feature selection)
corr_abs = corr_matrix.abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
    .sort_values("abs_corr", ascending=False)
)
print("Top 10 highly correlated feature pairs (sampled subset):")
high_corr_pairs.head(10)


**Finding:** Correlation is generally moderate across most sampled features, with a handful of pairs (mostly within the aggregated feature block) showing high correlation. This is expected since aggregated features are neighborhood statistics of the same underlying local features. We keep all features for the tree-based models (which handle redundancy natively) but this analysis would guide feature selection for the linear model.


### 3.5 Graph Structure Analysis

In [ ]:
# Build the transaction graph
G = nx.DiGraph()
G.add_edges_from(edges_df[["txId1", "txId2"]].values)

n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
n_weakly_connected = nx.number_weakly_connected_components(G)

print(f"Nodes (transactions with an edge): {n_nodes:,}")
print(f"Edges                            : {n_edges:,}")
print(f"Graph density                    : {density:.8f}")
print(f"Weakly connected components      : {n_weakly_connected:,}")


In [ ]:
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
total_deg = {n: in_deg.get(n, 0) + out_deg.get(n, 0) for n in G.nodes()}

degree_df = pd.DataFrame({
    "txId": list(total_deg.keys()),
    "in_degree": [in_deg[n] for n in total_deg.keys()],
    "out_degree": [out_deg[n] for n in total_deg.keys()],
    "total_degree": list(total_deg.values())
})

print(degree_df[["in_degree", "out_degree", "total_degree"]].describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

for ax, col, title, color in zip(
    axes,
    ["in_degree", "out_degree", "total_degree"],
    ["In-Degree Distribution", "Out-Degree Distribution", "Total Degree Distribution"],
    ["#2E86AB", "#F4A261", "#6A4C93"]
):
    sns.histplot(degree_df[col], bins=50, ax=ax, color=color, log_scale=(False, True))
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Degree")
    ax.set_ylabel("Count (log scale)")

plt.suptitle("Transaction Graph — Degree Distribution", fontsize=16, fontweight="bold", y=1.03)
plt.tight_layout()
save_fig(fig, "04_graph_degree_distribution.png")
plt.show()


**Finding:** The transaction graph shows a typical **heavy-tailed degree distribution** — most transactions connect to only 1–2 others, while a small number of "hub" transactions have very high degree. This power-law-like pattern is common in financial/blockchain networks and motivates graph-based features (degree, neighborhood risk) as useful signals for fraud detection, since illicit actors may exhibit unusual connectivity patterns (e.g., high fan-out to obscure trails, or clustering in mixing services).


In [ ]:
# Merge degree info with illicit label to check whether illicit transactions have different connectivity
degree_labeled = degree_df.merge(labeled_data[["txId", "label_name"]], on="txId", how="inner")

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=degree_labeled, x="label_name", y="total_degree", order=["licit", "illicit"],
    palette=[PALETTE["licit"], PALETTE["illicit"]], showfliers=False, ax=ax
)
ax.set_title("Total Degree — Illicit vs Licit Transactions", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Total Degree")
plt.tight_layout()
save_fig(fig, "04b_degree_by_class.png")
plt.show()


### 3.6 Time-Step Analysis

In [ ]:
time_step_counts = data.groupby(["time_step", "label_name"]).size().unstack(fill_value=0)
time_step_counts = time_step_counts.reindex(columns=["licit", "illicit", "unknown"], fill_value=0)

fig, ax = plt.subplots(figsize=(18, 6))
time_step_counts.plot(
    kind="bar", stacked=True, ax=ax,
    color=[PALETTE["licit"], PALETTE["illicit"], PALETTE["unknown"]], width=0.85
)
ax.set_title("Transaction Volume by Time Step and Class", fontweight="bold", fontsize=15)
ax.set_xlabel("Time Step")
ax.set_ylabel("Number of Transactions")
ax.legend(title="Class")
plt.xticks(rotation=90)
plt.tight_layout()
save_fig(fig, "05_timestep_class_distribution.png")
plt.show()


In [ ]:
# Illicit ratio over time (within labeled data only)
illicit_ratio_over_time = (
    labeled_data.groupby("time_step")["label"]
    .mean()
    .rename("illicit_ratio")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(16, 5.5))
ax.plot(illicit_ratio_over_time["time_step"], illicit_ratio_over_time["illicit_ratio"],
        marker="o", color=PALETTE["illicit"], linewidth=2)
ax.set_title("Illicit Transaction Ratio Over Time (Labeled Data Only)", fontweight="bold")
ax.set_xlabel("Time Step")
ax.set_ylabel("Illicit Ratio")
ax.axhline(labeled_data['label'].mean(), color="gray", linestyle="--", linewidth=1, label="Overall Average")
ax.legend()
plt.tight_layout()
save_fig(fig, "05b_illicit_ratio_over_time.png")
plt.show()


**Finding:** The illicit ratio fluctuates across time steps rather than staying constant, with some steps showing sharp spikes (consistent with known dark-market-related events in the original Elliptic dataset around certain time windows). This supports using a **time-aware split** for training/validation/testing rather than a naive random split, so the model is evaluated on genuinely "future" transactions.


## 4. Data Cleaning & Preprocessing

Based on the EDA above:
- No null values require imputation.
- We explicitly separate the **labeled** subset (licit/illicit) used for supervised training from the **unknown** subset used only for inference/scoring.
- We check and remove any duplicate transaction records.
- We verify feature data types and ranges are consistent for modeling.


In [ ]:
# ---- Duplicate check ----
n_dupes = data.duplicated(subset="txId").sum()
print(f"Duplicate txId rows: {n_dupes}")

if n_dupes > 0:
    data = data.drop_duplicates(subset="txId").reset_index(drop=True)
    print("Duplicates removed. New shape:", data.shape)
else:
    print("No duplicate transactions found — dataset is already unique per txId.")


In [ ]:
# ---- Data type / range sanity check ----
numeric_feature_cols = [c for c in data.columns if c.startswith("local_") or c.startswith("agg_")]
non_numeric = data[numeric_feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric feature columns (should be empty):", non_numeric)

print("\nFeature value range check:")
print(data[numeric_feature_cols].describe().T[["min", "max", "mean", "std"]].head(10))


In [ ]:
# ---- Split labeled vs unknown transactions ----
labeled_df = data[data["label"] != -1].reset_index(drop=True)
unknown_df = data[data["label"] == -1].reset_index(drop=True)

print(f"Labeled transactions (used for training/eval): {labeled_df.shape[0]:,}")
print(f"Unknown transactions (scored at inference only): {unknown_df.shape[0]:,}")
print(f"\nLabeled class balance:\n{labeled_df['label_name'].value_counts()}")


**Cleaning Summary:** The dataset required minimal cleaning — no missing cell values and no duplicate transaction IDs were found. The primary "cleaning" decision is a **modeling design choice**: excluding `unknown` transactions from supervised training while retaining them for downstream risk scoring, since they represent nodes the original labelers could not confidently classify (not corrupted or invalid data).


## 5. Feature Engineering (Including Basic Graph Features)

We enrich the raw 165 Elliptic features with lightweight **graph-derived features** computed from the transaction edgelist:
- `in_degree`, `out_degree`, `total_degree`
- `is_hub` — flag for unusually high-degree transactions (potential mixers/exchanges)

These graph features act as a cheap substitute for a full GNN, since the Elliptic aggregated features already partially encode neighborhood statistics.


In [ ]:
def add_graph_features(df: pd.DataFrame, graph: nx.DiGraph) -> pd.DataFrame:
    """Attach in/out/total degree and a hub flag to each transaction row."""
    in_d = dict(graph.in_degree())
    out_d = dict(graph.out_degree())

    df = df.copy()
    df["in_degree"] = df["txId"].map(in_d).fillna(0)
    df["out_degree"] = df["txId"].map(out_d).fillna(0)
    df["total_degree"] = df["in_degree"] + df["out_degree"]

    hub_threshold = df["total_degree"].quantile(0.99)
    df["is_hub"] = (df["total_degree"] >= hub_threshold).astype(int)
    return df


data = add_graph_features(data, G)
labeled_df = add_graph_features(labeled_df, G)
unknown_df = add_graph_features(unknown_df, G)

print("Graph features added: in_degree, out_degree, total_degree, is_hub")
labeled_df[["txId", "in_degree", "out_degree", "total_degree", "is_hub"]].head()


In [ ]:
# ---- Final feature set for modeling ----
GRAPH_FEATURES = ["in_degree", "out_degree", "total_degree", "is_hub"]
BASE_FEATURES = [c for c in data.columns if c.startswith("local_") or c.startswith("agg_")]
FEATURE_COLUMNS = BASE_FEATURES + GRAPH_FEATURES

print(f"Base Elliptic features : {len(BASE_FEATURES)}")
print(f"Graph-derived features : {len(GRAPH_FEATURES)}")
print(f"Total feature count    : {len(FEATURE_COLUMNS)}")


In [ ]:
X = labeled_df[FEATURE_COLUMNS].values
y = labeled_df["label"].values
time_steps = labeled_df["time_step"].values

print("Feature matrix X:", X.shape)
print("Label vector y:", y.shape)


## 6. Machine Learning Pipeline (Training + Evaluation)

### 6.1 Train / Validation / Test Split (Time-Aware)

Following the time-step analysis in Section 3.6, we split **chronologically**:
- **Train**: earliest ~60% of time steps
- **Validation**: next ~15% of time steps
- **Test**: final ~25% of time steps (most "future-like" and hardest to leak into)

This mirrors realistic deployment, where the model only ever sees past transactions during training.


In [ ]:
unique_steps = np.sort(labeled_df["time_step"].unique())
n_steps = len(unique_steps)

train_cutoff = unique_steps[int(n_steps * 0.60)]
val_cutoff   = unique_steps[int(n_steps * 0.75)]

train_mask = labeled_df["time_step"] <= train_cutoff
val_mask   = (labeled_df["time_step"] > train_cutoff) & (labeled_df["time_step"] <= val_cutoff)
test_mask  = labeled_df["time_step"] > val_cutoff

X_train, y_train = X[train_mask.values], y[train_mask.values]
X_val, y_val     = X[val_mask.values], y[val_mask.values]
X_test, y_test   = X[test_mask.values], y[test_mask.values]

print(f"Train: {X_train.shape[0]:,} rows | illicit ratio: {y_train.mean():.4f}")
print(f"Val  : {X_val.shape[0]:,} rows | illicit ratio: {y_val.mean():.4f}")
print(f"Test : {X_test.shape[0]:,} rows | illicit ratio: {y_test.mean():.4f}")


In [ ]:
# ---- Feature scaling (fit on train only, applied to all splits) ----
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Feature scaling complete (StandardScaler fit on training data only).")


### 6.2 Handling Class Imbalance

Illicit transactions are the minority class in the labeled set. We use class weighting (`class_weight='balanced'` / `scale_pos_weight`) rather than naive oversampling, to avoid synthetic-sample leakage across the time-based split.


In [ ]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / max(n_pos, 1)

print(f"Training set — illicit (pos): {n_pos:,} | licit (neg): {n_neg:,}")
print(f"scale_pos_weight for boosting models: {scale_pos_weight:.2f}")


### 6.3 Model Training

We train four models for comparison:
1. **Logistic Regression** — interpretable linear baseline
2. **Random Forest** — non-linear bagging baseline
3. **XGBoost** — gradient boosting
4. **LightGBM** — gradient boosting (typically fastest, strong on tabular data)


In [ ]:
def evaluate_model(model, X_eval, y_eval, model_name="Model", threshold=0.5):
    """Compute standard classification metrics with focus on the illicit (positive) class."""
    y_proba = model.predict_proba(X_eval)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision_illicit": precision_score(y_eval, y_pred, pos_label=1),
        "recall_illicit": recall_score(y_eval, y_pred, pos_label=1),
        "f1_illicit": f1_score(y_eval, y_pred, pos_label=1),
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "pr_auc": average_precision_score(y_eval, y_proba),
    }
    return metrics, y_proba, y_pred


In [ ]:
print("Training Logistic Regression...")
log_reg = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
)
log_reg.fit(X_train_scaled, y_train)
print("Done.")


In [ ]:
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=400, max_depth=14, class_weight="balanced",
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
print("Done.")


In [ ]:
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr", random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    verbose=False
)
print("Done.")


In [ ]:
print("Training LightGBM...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1
)
lgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_val_scaled, y_val)],
    eval_metric="average_precision",
    callbacks=[lgb.early_stopping(50, verbose=False)]
)
print("Done.")


### 6.4 Validation Set Evaluation

In [ ]:
models = {
    "Logistic Regression": log_reg,
    "Random Forest": rf_model,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
}

val_results = []
val_probas = {}
for name, model in models.items():
    metrics, proba, _ = evaluate_model(model, X_val_scaled, y_val, model_name=name)
    val_results.append(metrics)
    val_probas[name] = proba

val_results_df = pd.DataFrame(val_results).set_index("model").round(4)
val_results_df.sort_values("pr_auc", ascending=False)


## 7. Model Comparison & Best Model Selection

We compare all four models on the **validation set** using PR-AUC and F1 (illicit class) as the primary criteria — accuracy is misleading here due to class imbalance. The best model is then confirmed on the held-out **test set**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics_to_plot = ["precision_illicit", "recall_illicit", "f1_illicit", "roc_auc", "pr_auc"]
plot_df = val_results_df[metrics_to_plot].reset_index().melt(id_vars="model", var_name="metric", value_name="score")

sns.barplot(data=plot_df, x="metric", y="score", hue="model", ax=axes[0])
axes[0].set_title("Validation Metrics by Model", fontweight="bold")
axes[0].set_ylabel("Score")
axes[0].set_xlabel("")
axes[0].legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
axes[0].tick_params(axis='x', rotation=20)

# ROC curves
for name, model in models.items():
    fpr, tpr, _ = roc_curve(y_val, val_probas[name])
    auc_score = roc_auc_score(y_val, val_probas[name])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc_score:.3f})", linewidth=2)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[1].set_title("ROC Curves — Validation Set", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right", fontsize=10)

plt.tight_layout()
save_fig(fig, "07_roc_curve.png")
plt.show()


In [ ]:
# ---- Select best model by validation PR-AUC (most informative under imbalance) ----
best_model_name = val_results_df["pr_auc"].idxmax()
best_model = models[best_model_name]

print(f"Best model selected (by validation PR-AUC): {best_model_name}")
print(val_results_df.loc[best_model_name])


### 7.1 Final Evaluation on Held-Out Test Set

In [ ]:
test_metrics, test_proba, test_pred = evaluate_model(best_model, X_test_scaled, y_test, model_name=best_model_name)

print(f"=== {best_model_name} — Test Set Performance ===")
for k, v in test_metrics.items():
    if k != "model":
        print(f"{k:20s}: {v:.4f}")

print("\nFull classification report:")
print(classification_report(y_test, test_pred, target_names=["licit", "illicit"], digits=4))


In [ ]:
# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, test_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", cbar=True,
    xticklabels=["licit", "illicit"], yticklabels=["licit", "illicit"], ax=ax,
    annot_kws={"fontsize": 14, "fontweight": "bold"}
)
ax.set_title(f"Confusion Matrix — {best_model_name} (Test Set)", fontweight="bold")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
save_fig(fig, "06_confusion_matrix.png")
plt.show()


In [ ]:
# ---- Precision-Recall Curve ----
precisions, recalls, thresholds = precision_recall_curve(y_test, test_proba)
pr_auc = average_precision_score(y_test, test_proba)

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.plot(recalls, precisions, color=PALETTE["illicit"], linewidth=2.5, label=f"PR-AUC = {pr_auc:.4f}")
ax.fill_between(recalls, precisions, alpha=0.15, color=PALETTE["illicit"])
ax.set_title(f"Precision-Recall Curve — {best_model_name} (Test Set)", fontweight="bold")
ax.set_xlabel("Recall (Illicit)")
ax.set_ylabel("Precision (Illicit)")
ax.legend(loc="lower left")
plt.tight_layout()
save_fig(fig, "07b_precision_recall_curve.png")
plt.show()


In [ ]:
# ---- Optimal F1 threshold from PR curve ----
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])
DECISION_THRESHOLD = float(thresholds[best_idx])

print(f"Best F1 threshold: {DECISION_THRESHOLD:.4f}")
print(f"Precision @ threshold: {precisions[best_idx]:.4f}")
print(f"Recall @ threshold   : {recalls[best_idx]:.4f}")
print(f"F1 @ threshold       : {f1_scores[best_idx]:.4f}")


### 7.2 Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False).head(20)
elif hasattr(best_model, "coef_"):
    importance_df = pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "importance": np.abs(best_model.coef_[0])
    }).sort_values("importance", ascending=False).head(20)
else:
    importance_df = pd.DataFrame(columns=["feature", "importance"])

fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(data=importance_df, x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title(f"Top 20 Feature Importances — {best_model_name}", fontweight="bold")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
plt.tight_layout()
save_fig(fig, "05_model_feature_importance.png")
plt.show()


**Model Selection Summary:** All four models are compared on validation PR-AUC, precision/recall/F1 for the illicit class, and ROC-AUC. The best-performing model by these fraud-relevant metrics is promoted to the final pipeline and confirmed on the untouched test set above. Graph-derived degree features and a subset of aggregated neighborhood features consistently rank among the most important predictors, reinforcing the value of network structure in this task.


## 8. Risk Scoring Function

A reusable function that converts model probability into a clean **0–1 risk score**, a categorical **risk band**, and a **confidence** measure, while properly handling the `unknown` class at inference time.


In [ ]:
def compute_risk_band(score: float) -> str:
    """Map a 0-1 risk score to a human-readable risk band."""
    if score >= 0.75:
        return "HIGH"
    elif score >= 0.45:
        return "MEDIUM"
    elif score >= 0.20:
        return "LOW"
    else:
        return "MINIMAL"


def compute_confidence(score: float) -> float:
    """
    Confidence is highest when the score is far from the decision threshold
    (i.e., the model is not 'on the fence'), scaled to 0-1.
    """
    distance = abs(score - DECISION_THRESHOLD)
    max_distance = max(DECISION_THRESHOLD, 1 - DECISION_THRESHOLD)
    return float(round(min(distance / max_distance, 1.0), 4))


def get_risk_score(feature_row, model=best_model, scaler=scaler) -> dict:
    """
    Compute a 0-1 risk score, prediction, risk band, and confidence for a
    single transaction/wallet given its raw (unscaled) feature vector,
    ordered as FEATURE_COLUMNS.
    """
    feature_row = np.asarray(feature_row, dtype=float).reshape(1, -1)
    scaled_row = scaler.transform(feature_row)
    risk_score = float(model.predict_proba(scaled_row)[0, 1])
    predicted_label = "illicit" if risk_score >= DECISION_THRESHOLD else "licit"

    return {
        "risk_score": round(risk_score, 4),
        "risk_band": compute_risk_band(risk_score),
        "predicted_label": predicted_label,
        "confidence": compute_confidence(risk_score),
        "threshold_used": round(DECISION_THRESHOLD, 4),
    }


# Quick sanity check on a known test sample
sample_features = X_test[0]
print("Sample risk assessment:", get_risk_score(sample_features))


In [ ]:
# ---- Score the full labeled + unknown dataset for downstream use ----
all_X = data[FEATURE_COLUMNS].values
all_X_scaled = scaler.transform(all_X)
data["risk_score"] = best_model.predict_proba(all_X_scaled)[:, 1]
data["risk_band"] = data["risk_score"].apply(compute_risk_band)

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=data, x="risk_score", hue="label_name", bins=50, palette=PALETTE,
             element="step", stat="density", common_norm=False, ax=ax)
ax.axvline(DECISION_THRESHOLD, color="black", linestyle="--", linewidth=1.5, label=f"Threshold={DECISION_THRESHOLD:.2f}")
ax.set_title("Risk Score Distribution by True Class (All Transactions)", fontweight="bold")
ax.set_xlabel("Risk Score")
ax.set_ylabel("Density")
ax.legend(title="True Class")
plt.tight_layout()
save_fig(fig, "08_risk_score_distribution.png")
plt.show()


**Finding:** The risk score distribution clearly separates illicit from licit transactions, with most licit transactions scoring low and illicit transactions concentrated at higher scores. `unknown` transactions span the full range, as expected — the model surfaces a data-driven prior risk estimate for them even though ground truth is unavailable.


## 9. Final Inference Pipeline

A single reusable function that takes a **new transaction's raw features** (and optionally a known `txId` for graph context) and returns predicted class, risk score, and confidence — ready for integration with other systems (e.g., merging with a Cash pipeline).


In [ ]:
txid_to_risk = dict(zip(data["txId"], data["risk_score"]))

def neighbor_risk_profile(tx_id, graph=G, risk_lookup=txid_to_risk, hops: int = 2) -> dict:
    """Summarize the risk profile of a transaction's graph neighbors up to `hops` hops."""
    if tx_id not in graph:
        return {"neighbors_found": 0, "avg_neighbor_risk": None,
                "max_neighbor_risk": None, "high_risk_neighbor_ratio": None}

    neighbors, frontier = set(), {tx_id}
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(graph.predecessors(node))
            next_frontier.update(graph.successors(node))
        neighbors.update(next_frontier)
        frontier = next_frontier
    neighbors.discard(tx_id)

    scores = np.array([risk_lookup[n] for n in neighbors if n in risk_lookup])
    if scores.size == 0:
        return {"neighbors_found": 0, "avg_neighbor_risk": None,
                "max_neighbor_risk": None, "high_risk_neighbor_ratio": None}

    return {
        "neighbors_found": int(scores.size),
        "avg_neighbor_risk": round(float(scores.mean()), 4),
        "max_neighbor_risk": round(float(scores.max()), 4),
        "high_risk_neighbor_ratio": round(float((scores >= 0.75).mean()), 4),
    }


def run_inference(feature_row, tx_id=None, model=best_model, scaler=scaler, graph=G) -> dict:
    """
    End-to-end inference for a single transaction/wallet.

    Returns predicted class, risk score, risk band, confidence, and (if a known
    txId is given) graph-based neighbor risk context for basic wallet attribution.
    """
    result = get_risk_score(feature_row, model=model, scaler=scaler)
    result["tx_id"] = tx_id

    if tx_id is not None:
        result["graph_context"] = neighbor_risk_profile(tx_id, graph=graph, hops=2)
    else:
        result["graph_context"] = "No txId provided — graph context unavailable"

    return result

print("Inference pipeline ready: run_inference(feature_row, tx_id=None)")


### Example: Score New Transactions

In [ ]:
# --- Example 1: an existing labeled transaction (with graph context) ---
example_id = labeled_df["txId"].iloc[25]
example_features = labeled_df.loc[labeled_df["txId"] == example_id, FEATURE_COLUMNS].values[0]

result_1 = run_inference(example_features, tx_id=example_id)
print("Example 1 — existing transaction with graph context:")
result_1


In [ ]:
# --- Example 2: an unknown-class transaction (scored without ground truth) ---
example_unknown_id = unknown_df["txId"].iloc[0]
example_unknown_features = unknown_df.loc[unknown_df["txId"] == example_unknown_id, FEATURE_COLUMNS].values[0]

result_2 = run_inference(example_unknown_features, tx_id=example_unknown_id)
print("Example 2 — 'unknown' class transaction:")
result_2


In [ ]:
# --- Example 3: a brand-new/simulated transaction (no txId, no graph context) ---
np.random.seed(7)
new_tx_features = example_features + np.random.normal(0, 0.05, size=example_features.shape)

result_3 = run_inference(new_tx_features, tx_id=None)
print("Example 3 — new/unseen transaction (no graph context):")
result_3


## 10. Save Model, Pipeline & Summary

We persist the best model, scaler, feature order, and decision threshold as a single bundle for reuse, plus a text summary report of the full run.


In [ ]:
artifact_bundle = {
    "model": best_model,
    "model_name": best_model_name,
    "scaler": scaler,
    "feature_columns": FEATURE_COLUMNS,
    "decision_threshold": DECISION_THRESHOLD,
    "test_metrics": test_metrics,
}

MODEL_PATH = os.path.join(MODEL_DIR, "crypto_fraud_best_model_pipeline.joblib")
joblib.dump(artifact_bundle, MODEL_PATH)

try:
    GRAPH_PATH = os.path.join(MODEL_DIR, "crypto_tx_graph.joblib")
    joblib.dump(G, GRAPH_PATH)
    print("Saved transaction graph to:", GRAPH_PATH)
except Exception as e:
    print("Graph save skipped:", e)

print("Saved model bundle to:", MODEL_PATH)


In [ ]:
# ---- Reload check ----
loaded_bundle = joblib.load(MODEL_PATH)
loaded_model = loaded_bundle["model"]
loaded_scaler = loaded_bundle["scaler"]
loaded_threshold = loaded_bundle["decision_threshold"]

print("Reloaded model:", loaded_bundle["model_name"])
print("Reloaded decision threshold:", loaded_threshold)

sanity_check = get_risk_score(example_features, model=loaded_model, scaler=loaded_scaler)
print("Sanity check risk score (reloaded model):", sanity_check)


In [ ]:
# ---- Write a plain-text summary report ----
summary_lines = [
    "=" * 60,
    "CRYPTO FRAUD DETECTION PIPELINE — RUN SUMMARY",
    "=" * 60,
    f"Total transactions          : {data.shape[0]:,}",
    f"Labeled transactions        : {labeled_df.shape[0]:,}",
    f"Unknown transactions        : {unknown_df.shape[0]:,}",
    f"Illicit ratio (labeled)     : {labeled_df['label'].mean():.4f}",
    f"Total features used         : {len(FEATURE_COLUMNS)}",
    "",
    "Model Comparison (Validation Set):",
    val_results_df.round(4).to_string(),
    "",
    f"Best Model Selected         : {best_model_name}",
    f"Decision Threshold          : {DECISION_THRESHOLD:.4f}",
    "",
    "Test Set Performance:",
] + [f"  {k:20s}: {v:.4f}" for k, v in test_metrics.items() if k != "model"] + [
    "",
    f"Model bundle saved to       : {MODEL_PATH}",
    f"Images saved to             : {IMAGES_DIR}",
    "=" * 60,
]

summary_text = "\n".join(summary_lines)
print(summary_text)

SUMMARY_PATH = os.path.join(MODEL_DIR, "pipeline_run_summary.txt")
with open(SUMMARY_PATH, "w") as f:
    f.write(summary_text)
print(f"\nSummary report saved to: {SUMMARY_PATH}")


In [ ]:
# ---- List all saved images ----
saved_images = sorted(os.listdir(IMAGES_DIR))
print(f"Total images saved: {len(saved_images)}")
for img in saved_images:
    print(" -", img)


---
### ✅ Pipeline Summary

| Component | Output |
|---|---|
| EDA | Class distribution, missing values, feature comparisons, correlation heatmap, graph structure, time-step analysis — all saved as PNGs |
| Cleaning | Duplicate check, type/range validation, labeled vs unknown split |
| Feature Engineering | 165 base features + 4 graph-derived features (`in_degree`, `out_degree`, `total_degree`, `is_hub`) |
| Modeling | Logistic Regression, Random Forest, XGBoost, LightGBM — compared on validation PR-AUC/F1/ROC-AUC |
| Best Model | Selected automatically and confirmed on held-out test set |
| Risk Scoring | `get_risk_score()` → 0–1 score, risk band, confidence |
| Wallet/Tx Attribution | `neighbor_risk_profile()` → 1/2-hop high-risk neighbor exposure |
| Inference | `run_inference()` → single entry point combining model + graph context |
| Persistence | `crypto_fraud_best_model_pipeline.joblib` + transaction graph + text summary report |

**Next step:** The `run_inference()` output schema (`predicted_label`, `risk_score`, `risk_band`, `confidence`, `graph_context`) is designed to merge cleanly with the Cash pipeline's output for a unified cross-channel fraud risk view.
